In [1]:
# (C) Martin Reißel

from sympy import *

from IPython.display import display, Math, Latex
from sympy.interactive import printing
printing.init_printing(use_latex='mathjax')
platex = lambda A: latex(A, mat_str='pmatrix', mat_delim='')

import pylab as pl

# Aufgabenstellung

Approximieren Sie das Integral

In [2]:
x = symbols('x')
f = Lambda(x, sin(pi/12* x)**2)

a = 0
b = 12

Iex = Integral(f(x), (x,a,b))

Math(latex(Iex))

<IPython.core.display.Math object>

mit Hilfe

* der summierten Trapezregel $T_{m_0}$, $T_{m_1}$, $T_{m_2}$, $T_{m_3}$ mit
$m_0=1$, $m_1=2$, $m_2=3$ bzw. $m_3=4$ Trapezen.

* des Extrapolationsverfahrens nach Romberg.

* des Extrapolationsverfahrens nach Bulirsch.

Extrapolieren Sie bis Ordnung $6$. Benutzen Sie als Startwerte für die Extrapolation
die Ergebnisse aus dem ersten Teil.

# Lösung

## Exakter Wert (ist in der Aufgabe nicht gefragt)

In [3]:
iex = Iex.doit()
Math(latex(Iex) + '=' + latex(iex) + r'\approx ' + latex(iex.evalf(7)))

<IPython.core.display.Math object>

## Trapezregel

Für die summierte Trapezregel gilt

$$
T_m(f) = \frac{h}{2} \bigl(f(x_0) + 2 f(x_1) + \ldots + 2 f(x_{m-1}) + f(x_m)\bigr),
\qquad
h = \frac{b-a}{m},
\quad
x_j = a + j\: h.
$$

Für die $T_{m_i}$ erhält man damit

In [4]:
# summierte Trapezregel
def tsum(f, a, b, m):
    h = (b - a) / Rational(m)
    
    w = f(a)
    for i in range(1, m):
        w += 2 * f(a + i*h)
    w += f(b)
    
    return w*h/Rational(2)

m = [1, 2, 3, 4]
T = []
for mi in m:
    T.append(tsum(f, a, b, mi))
    
Math(latex(T) + ".")

for mi,Ti in zip(m,T):
    display(Math('m = {} \qquad T = {} \qquad h = {}'.format(latex(mi), latex(Ti), latex(Rational(b-a, mi)))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Extrapolation

Sei $T$ die zu $q$ Schrittweiten $h_0 > h_1 > \ldots >h_{q-1}$
extrapolierte äquidistante summierte Trapez-Regel. Dann ist $T =
P_{q-1 q-1}$ mit
\begin{align*}
P_{i0} &= T_{m_i} && i = 0, \ldots , q-1\\[1ex]
P_{ij} &= P_{ij-1} + \frac{P_{ij-1} -
P_{i-1j-1}}{(\frac{h_{i-j}}{h_i})^2 - 1} && 1 \le j \le i \le q-1.
\end{align*}
$T$ approximiert $\int\limits_a^b f(t) dt$ für $f \in C^{2q-1}
[a,b]$ mit Ordnung $2q$.

Um Ordnung 6 zu erreichen muss man also mit 3 verschiedenen Trapezbreiten arbeiten.

In [5]:
# Extrapolationstableau
def extra(hmi, Tmi):
    q = len(hmi)
    P = zeros(q)
    
    P[:,0] = Matrix(Tmi)
    for i in range(1,q):
        for j in range(1,i+1):
            P[i, j] = P[i, j-1] + (P[i, j-1] - P[i-1, j-1]) / ((hmi[i-j] / hmi[i])**2 - 1)
    
    return P

### Romberg

Bei Romberg startet man mit einem Trapez und verdoppelt dann die Anzahl der Trapeze in jeden weiteren Schritt.

In [6]:
# Romberg
mi  = [1, 2, 4]
hmi  = [(b - a) / Rational(m)  for m in mi]
Tmi = [tsum(f, a, b, m)  for m in mi]

P = extra(hmi, Tmi)

display(Math(r'm_i = ' + latex(mi)))
display(Math(r'h_i = ' + latex(hmi)))
display(Math(r'T_{m_i} = ' + latex(Tmi) + r'\approx' + latex(Matrix(Tmi).T.evalf(7))))
display(Math(r'P = ' + latex(P) + r'\approx' + latex(P.evalf(7))))

T = P[-1, -1]

Math('Romberg = '+ latex(T.simplify()) + r' \approx ' + latex(T.evalf(7)) )

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Bulirsch

Bei Bulirsch benutzen wir 1,2 und 3 Trapeze.

In [7]:
# Bulirsch
mi  = [1, 2, 3]
hmi  = [(b - a) / Rational(m)  for m in mi]
Tmi = [tsum(f, a, b, m)  for m in mi]

P = extra(hmi, Tmi)

display(Math(r'm_i = ' + latex(mi)))
display(Math(r'h_i = ' + latex(hmi)))
display(Math(r'T_{m_i} = ' + latex(Tmi) + r'\approx' + latex(Matrix(Tmi).T.evalf(7))))
display(Math(r'P = ' + latex(P) + r'\approx' + latex(P.evalf(7))))

T = P[-1, -1]

Math('Bulirsch = '+ latex(T.simplify()) + r' \approx ' + latex(T.evalf(7)) )

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>